In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# **1. Load preprocessed datasets**

In [2]:
X_train = pd.read_csv('X_train_processed.csv')
y_train = pd.read_csv('y_train.csv')

X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')

In [3]:
print("Data loaded successfully.")

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

Data loaded successfully.
X_train shape: (614, 12)
y_train shape: (614, 2)
X_test shape: (154, 12)
y_test shape: (154, 2)


In [4]:
X_train.head(10)

,relative_compactness,surface_area,wall_area,roof_area,overall_height,orientation,glazing_area,glazing_distribution,envelope_surface_ratio,wall_roof_ratio,surface_to_volume,aspect_efficiency
0,0.553671,-0.696222,-0.007372,-0.679048,7.0,2,0.10,1,0.331203,0.331203,-1.245941,0.752863
1,-1.155118,1.250664,0.558439,0.957063,3.5,4,0.40,2,-0.593690,-0.593690,1.190185,-0.966335
2,0.933402,-0.974349,-0.573184,-0.679048,7.0,4,0.25,2,0.078960,0.078960,-1.383835,0.752863
3,1.313133,-1.252476,-0.007372,-1.224418,7.0,4,0.25,1,0.987036,0.987036,-0.887417,1.165471
4,-0.965252,0.972537,-0.007372,0.957063,3.5,5,0.10,4,-0.761852,-0.761852,1.006326,-0.966335
5,1.313133,-1.252476,-0.007372,-1.224418,7.0,2,0.25,2,0.987036,0.987036,-0.887417,1.165471
6,1.313133,-1.252476,-0.007372,-1.224418,7.0,5,0.40,1,0.987036,0.987036,-0.887417,1.165471
7,0.268873,-0.418096,0.558439,-0.679048,7.0,4,0.25,1,0.583447,0.583447,-1.108047,0.752863
8,-1.344983,1.528790,1.124251,0.957063,3.5,4,0.40,4,-0.425527,-0.425527,1.374043,-0.966335
9,-1.155118,1.250664,0.558439,0.957063,3.5,4,0.10,1,-0.593690,-0.593690,1.190185,-0.966335


# **2. Train and Validate on different models**

In [5]:
# Create a dictionary of models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "K-Neighbors": KNeighborsRegressor(),
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": LGBMRegressor(random_state=42, verbose=-1),
    "Neural Network": MLPRegressor(random_state=42, max_iter=2000)
}

In [6]:
results = []

print("Training and validating models...")

# Define target column names for better readability
target_names = y_train.columns.tolist()

for name, model in models.items():
    print(f"Training {name}...")

    # For models that don't natively support multi-output, wrap them with MultiOutputRegressor
    # LinearRegression and MLPRegressor typically handle multi-output natively.
    current_model = model
    if not isinstance(model, (LinearRegression, MLPRegressor)):
        current_model = MultiOutputRegressor(model)

    # Train the model with the 2D target array
    current_model.fit(X_train, y_train.values)

    # Make predictions on the test set
    y_pred = current_model.predict(X_test)

    # Initialize a dictionary to store metrics for the current model
    model_metrics = {"Model": name}

    # Calculate regression metrics for each target variable
    mae_scores = []
    mse_scores = []
    rmse_scores = []
    r2_scores = []
    mape_scores = []

    for i, target_name in enumerate(target_names):
        # Metrics for individual target
        mae = mean_absolute_error(y_test.values[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.values[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test.values[:, i], y_pred[:, i])

        # Handle division by zero for MAPE if actual values are 0, though unlikely for energy load
        # Using a small epsilon to avoid RuntimeWarning for division by zero in MAPE calculation
        mape = np.mean(np.abs((y_test.values[:, i] - y_pred[:, i]) / (y_test.values[:, i] + 1e-8))) * 100

        model_metrics[f"Validation MAE ({target_name})"] = mae
        model_metrics[f"Validation MSE ({target_name})"] = mse
        model_metrics[f"Validation RMSE ({target_name})"] = rmse
        model_metrics[f"Validation R-squared ({target_name})"] = r2
        model_metrics[f"Validation MAPE ({target_name})"] = mape

        mae_scores.append(mae)
        mse_scores.append(mse)
        rmse_scores.append(rmse)
        r2_scores.append(r2)
        mape_scores.append(mape)

    # Calculate and store metrics across both targets
    model_metrics["MAE"] = mae_scores
    model_metrics["MSE"] = mse_scores
    model_metrics["RMSE"] = rmse_scores
    model_metrics["R-squared"] = r2_scores
    model_metrics["MAPE"] = mape_scores

    # Calculate and store average metrics across both targets
    model_metrics["Mean MAE"] = np.mean(mae_scores)
    model_metrics["Mean MSE"] = np.mean(mse_scores)
    model_metrics["Mean RMSE"] = np.mean(rmse_scores)
    model_metrics["Mean R-squared"] = np.mean(r2_scores)
    model_metrics["Mean MAPE"] = np.mean(mape_scores)

    results.append(model_metrics)

print("Training complete.")

Training and validating models...
Training Linear Regression...
Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...
Training K-Neighbors...
Training XGBoost...
Training LightGBM...
Training Neural Network...
Training complete.


# **3. Compare Models to Find the Best One**

In [7]:
results_df = pd.DataFrame(results, columns=["Model", "MAE", "MSE", "RMSE", "R-squared", "MAPE", "Mean MAE", "Mean MSE", "Mean RMSE", "Mean R-squared", "Mean MAPE"])
results_df = results_df.sort_values(by="Mean R-squared", ascending=False) # higher R-squared value indicates a better regression model
print("Model Performance on Validation Set")
display(results_df)

Model Performance on Validation Set


,Model,MAE,MSE,RMSE,R-squared,MAPE,Mean MAE,Mean MSE,Mean RMSE,Mean R-squared,Mean MAPE
5,XGBoost,"[0.24990926110899286, 0.44960657144521715]","[0.13938696574953618, 0.9259039532935501]","[0.37334563844986346, 0.9622390312669458]","[0.9986627180918197, 0.9900072138485764]","[1.2365450071515678, 1.6055531309928799]",0.349758,0.532645,0.667792,0.994335,1.421049
6,LightGBM,"[0.3112446809876528, 0.6846449966576016]","[0.180565819586, 1.1715888798707625]","[0.42493037027964947, 1.0823995934361592]","[0.9982676471758344, 0.9873556678397476]","[1.410446486191071, 2.4046746258508755]",0.497945,0.676077,0.753665,0.992812,1.907561
3,Gradient Boosting,"[0.36657328608791595, 0.977305855516779]","[0.2362270996050662, 2.1406931588563896]","[0.48603199442533224, 1.4631107814709006]","[0.9977336315140729, 0.9768966436786717]","[1.6768362565360486, 3.506224659917657]",0.671940,1.188460,0.974571,0.987315,2.591530
2,Random Forest,"[0.3510655844155844, 1.0734272727272722]","[0.23759051461038763, 2.986615557012988]","[0.4874325744247994, 1.7281827325294592]","[0.997720550877658, 0.9677670556739877]","[1.4584579168499499, 3.544298782002338]",0.712246,1.612103,1.107808,0.982744,2.501378
7,Neural Network,"[0.6217453270779901, 1.1953498368073676]","[0.713651551088763, 3.1218982160572972]","[0.8447789954116774, 1.7668894181745776]","[0.9931532098221391, 0.9663070222903769]","[2.8067786652769566, 4.101767249695532]",0.908548,1.917775,1.305834,0.979730,3.454273
1,Decision Tree,"[0.42383116883116906, 1.1566883116883118]","[0.38491623376623413, 4.069692857142856]","[0.6204161778727519, 2.017347976216016]","[0.9963071043779991, 0.9560779816537716]","[1.7402334317295771, 4.071276333595086]",0.790260,2.227305,1.318882,0.976193,2.905755
0,Linear Regression,"[1.7824021604251918, 2.0086765187612436]","[5.169734961970652, 7.096239448482637]","[2.2737051176374328, 2.663876770513726]","[0.9504014382008086, 0.9234140830312434]","[9.33226764529057, 8.232114724663193]",1.895539,6.132987,2.468791,0.936908,8.782191
4,K-Neighbors,"[2.6898181818181826, 2.4950129870129873]","[13.209410025974028, 11.84160103896104]","[3.6344752064051873, 3.441162745201255]","[0.8732686019063562, 0.8721999334251705]","[12.575209054878474, 10.0317578198748]",2.592416,12.525506,3.537819,0.872734,11.303483


In [8]:
# Select the best model automatically
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print(f"Best Model Selected: {best_model_name}")

Best Model Selected: XGBoost


# **4. Retrain the Best Model on the Full Dataset**

In [9]:
X_full = pd.concat([X_train, X_test], axis=0)
y_full = pd.concat([y_train, y_test], axis=0)

print(f"X_full shape: {X_full.shape}")
print(f"y_full shape: {y_full.shape}")

X_full shape: (768, 12)
y_full shape: (768, 2)


In [10]:
initial_best_model = models[best_model_name]

# Check if the model needs to be wrapped with MultiOutputRegressor
# XGBoost typically handles multi-output regression natively, but the original code used MultiOutputRegressor for non-native ones.
# We will replicate the logic to ensure consistency.
if not isinstance(initial_best_model, (LinearRegression, MLPRegressor)):
    retrained_best_model = MultiOutputRegressor(initial_best_model)
else:
    retrained_best_model = initial_best_model

# Retrain the best model on the full dataset
retrained_best_model.fit(X_full, y_full.values)

print(f"Best model ({best_model_name}) successfully retrained on the full dataset.")

Best model (XGBoost) successfully retrained on the full dataset.


# **5. Save the best model**

**XGBoost** is the best overall model. It has the highest mean R-squared scores. This makes it the most reliable choice.

In [ ]:
# Save the retrained best model (fitted on the full dataset)
filename = f'best_energy_model_{best_model_name.replace(" ", "_")}.pkl'
joblib.dump(retrained_best_model, filename)

print(f"Best model saved as {filename}")

Best model saved as best_energy_model_XGBoost.pkl
